# CK-M worked solution: the model variant (the water tower)

*Energy Systems, Step by Step* · Modeler checkpoint CK-M. The formulation, the code, the answers to tasks 2 to 5 and 9 to 10, and the validation. `assert` cells check the checkpoint's numbers.

In [ ]:
import json, os
import numpy as np
import scipy
from scipy.optimize import linprog

print("NumPy", np.__version__, "· SciPy", scipy.__version__)
major, minor = (int(v) for v in scipy.__version__.split(".")[:2])
assert (major, minor) >= (1, 9), "this notebook needs SciPy 1.9 or later (pip install -U scipy)"

In [ ]:
# The water brief (illustrative). Hour t runs from t:00 to t+1:00.
water = [180, 160, 150, 150, 170, 250, 420, 610, 640, 560, 480, 450, 440, 430, 420, 430, 470, 560, 640, 650, 560, 430, 320, 230]        # m3 used in each hour
price = [0.06, 0.06, 0.06, 0.06, 0.06, 0.06, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.2, 0.2, 0.2, 0.2, 0.2, 0.1, 0.06, 0.06]   # $ per kWh
KWH_PER_M3 = 0.5        # kWh of electricity per m3 pumped
PUMP_MAX = 900          # m3 per hour
TOWER_MIN, TOWER_MAX = 800, 4000   # m3
TOWER_START = 2000       # m3 at the start of the day; at least as much at the end
DEMAND_CHARGE = 0.40   # $ per kW of the day's highest hourly draw
assert len(water) == len(price) == 24
print(f"{sum(water):,} m3 used in the day; highest hour {max(water)} m3")

## The formulation

Hours $t = 0, \dots, 23$. Decisions: water pumped $q_t$ (m³ in hour $t$), the tower's level at the end of hour $t$, $s_t$ (m³), and the day's highest draw $P$ (kW).

$$\min \; \sum_t \pi_t\, k\, q_t + c\,P \quad \text{s.t.} \quad s_t = s_{t-1} + q_t - w_t \;(s_{-1} = S_0),\;\; \underline S \le s_t \le \bar S,\;\; 0 \le q_t \le \bar Q,\;\; k\,q_t \le P,\;\; s_{23} \ge S_0.$$

Units: $\pi_t k q_t$ is \$/kWh × kWh/m³ × m³ = \$, and $cP$ is \$/kW × kW = \$. Each hour lasts one hour, so $k q_t$ in kWh is also the draw in kW. The highest draw is linearised like a built capacity: one variable that every hour's draw must stay under.

In [ ]:
def water_model(demand_charge=DEMAND_CHARGE, tower_max=TOWER_MAX, pump_max=PUMP_MAX):
    T = 24
    q = lambda t: t; s = lambda t: T + t; P = 2 * T            # columns: q_0..q_23, s_0..s_23, P
    c = np.zeros(2 * T + 1)
    c[:T] = [price[t] * KWH_PER_M3 for t in range(T)]
    c[P] = demand_charge
    bounds = [(0, pump_max)] * T + [(TOWER_MIN, tower_max)] * T + [(0, None)]
    A_eq, b_eq, A_ub, b_ub = [], [], [], []
    for t in range(T):
        r = np.zeros(2 * T + 1); r[s(t)] = 1; r[q(t)] = -1          # s_t - s_{t-1} - q_t = -w_t
        if t:
            r[s(t - 1)] = -1
        A_eq.append(r); b_eq.append((0 if t else TOWER_START) - water[t])
        r = np.zeros(2 * T + 1); r[q(t)] = KWH_PER_M3; r[P] = -1     # k q_t <= P
        A_ub.append(r); b_ub.append(0)
    r = np.zeros(2 * T + 1); r[s(T - 1)] = -1                          # s_23 >= S0
    A_ub.append(r); b_ub.append(-TOWER_START)
    res = linprog(c, A_ub=np.array(A_ub), b_ub=b_ub, A_eq=np.array(A_eq), b_eq=b_eq, bounds=bounds, method="highs")
    assert res.status == 0, res.message
    return dict(cost=res.fun, q=res.x[:T], s=res.x[T:2 * T], P=res.x[P], res=res)

best = water_model()
peak_hours = [t for t in range(24) if price[t] == max(price)]
energy = sum(best["q"][t] * KWH_PER_M3 * price[t] for t in range(24))
print(f"Least cost: ${best['cost']:.2f} = ${energy:.2f} energy + ${best['cost'] - energy:.2f} demand charge")
print(f"Highest draw: {best['P']:.1f} kW; pumped at the peak price: {sum(best['q'][t] for t in peak_hours):.1f} m3")
print("One least-cost schedule (m3 pumped):", np.round(best["q"]).astype(int).tolist())
assert abs(best["cost"] - 526.9) < 1e-6
assert abs(best["P"] - 255) < 1e-6
assert abs(sum(best["q"][t] for t in peak_hours) - 330) < 1e-4

**A larger tower** (task 5). Solve again with 100 m³ more room; compare with the duals of the tower's upper limits (SciPy returns them as the marginals of the upper bounds).

In [ ]:
bigger = water_model(tower_max=TOWER_MAX + 100)
saving = best["cost"] - bigger["cost"]
duals = -best["res"].upper.marginals[24:48].sum()
print(f"100 m3 more tower saves ${saving:.4f} a day; the upper-limit duals add up to ${duals:.6f} per m3")
assert abs(saving - 6.8333) < 1e-3 and abs(duals * 100 - saving) < 1e-6

## Validation (tasks 9 to 11)

Two bounds by hand: **pump as you go** is a feasible plan, so the optimum cannot cost more; **every m³ at the cheapest price, with the draw spread evenly** cannot be beaten. And one special case the model must reproduce exactly: with no demand charge and no limits on the tower or the pumps, everything is pumped at the night price.

In [ ]:
ub = sum(price[t] * KWH_PER_M3 * water[t] for t in range(24)) + DEMAND_CHARGE * KWH_PER_M3 * max(water)
lb = min(price) * KWH_PER_M3 * sum(water) + DEMAND_CHARGE * KWH_PER_M3 * sum(water) / 24
special = water_model(demand_charge=0, tower_max=1e9, pump_max=1e9)["cost"]
print(f"Lower bound ${lb:.2f} <= optimum ${best['cost']:.2f} <= upper bound ${ub:.2f}")
print(f"Special case: model ${special:.2f}, by hand ${min(price) * KWH_PER_M3 * sum(water):.2f}")
assert abs(ub - 731.8) < 1e-6 and abs(lb - 375.6667) < 1e-4 and lb <= best["cost"] <= ub
assert abs(special - 294) < 1e-6

**What these checks cannot show.** The bounds and the special case show the model is plausible; that its schedule is the cheapest rests on the formulation being right and on the solver's optimality. The schedule is not unique: other schedules cost exactly the same, so compare costs, not schedules.